# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. **`5 (2, 3)`** — `np.arange(6).reshape(2, 3)` даёт `[[0, 1, 2], [3, 4, 5]]`. `a[1, 2]` — строка 1, столбец 2 → `5`; форма — 2 строки × 3 столбца.
2. **`1 8`** — максимум `8` стоит на позициях 1 и 3. `argmax` возвращает индекс **первого** максимума → `1`; `max` → `8`.
3. **`99`** — `a[::3]` — это срез, а срез в NumPy — **view** (представление), а не копия: `b` смотрит на те же данные, что и `a`. Запись `b[0] = 99` меняет и `a[0]`.
4. **`2 0.5`** — `a > 3` даёт маску `[False, True, True, False]`. `sum()` считает `True` как 1 → `2`; `mean()` — доля `True` → `2/4 = 0.5`.
5. **`(2, 3) 0.25`** — `np.zeros((2, 3)).shape` → `(2, 3)`. `np.linspace(0, 1, 5)` → `[0, 0.25, 0.5, 0.75, 1]`, индекс 1 → `0.25`.

Проверка запуском (вывод должен совпасть с предсказаниями):

In [1]:
import numpy as np

# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

5 (2, 3)
1 8
99
2 0.5
(2, 3) 0.25


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [4]:
# База
# Вспомогательная функция: сумма с пробелом как разделителем тысяч
fmt = lambda x: f"{x:,.0f}".replace(",", " ")

# 1. Самый дорогой заказ
i = amount.argmax()
print(f"1. Позиция {i}: сумма {fmt(amount[i])} ₸, клиент {cust[i]}, город {city[i]}")
print(f"   Заказов с такой же максимальной суммой: {(amount == amount.max()).sum()} (argmax берёт первый)")

# 2. Выручка: маска paid и её отрицание ~paid
print(f"2. Оплаченные: {fmt(amount[paid].sum())} ₸ | неоплаченные: {fmt(amount[~paid].sum())} ₸")

# 3. Шымкент: сумма булевой маски = число True, среднее = доля True
shym = city == "Шымкент"
print(f"3. Заказов в Шымкенте: {shym.sum()} из {len(city)}, доля: {shym.mean():.1%}")

1. Позиция 1001: сумма 9 999 990 ₸, клиент TEST, город Алматы
   Заказов с такой же максимальной суммой: 3 (argmax берёт первый)
2. Оплаченные: 101 909 360 ₸ | неоплаченные: 7 974 000 ₸
3. Заказов в Шымкенте: 804 из 5463, доля: 14.7%


**Ответы [База]**

1. **9 999 990 ₸, клиент `TEST`, Алматы** (позиция 1001, заказ 105281; `argmax` берёт первый из трёх одинаковых). *Менеджеру:* это не реальная покупка, а тестовые заказы клиента TEST — их три, по 9 999 990 ₸, их нужно исключать из отчётов.
2. **Оплаченные — 101 909 360 ₸, неоплаченные — 7 974 000 ₸.** *Менеджеру:* в оплаченной выручке сидят ≈30 млн ₸ тестовых заказов TEST; без них реальная выручка — около 71,9 млн ₸.
3. **804 заказа, доля 14,7 %.** *Менеджеру:* примерно каждый седьмой заказ приходит из Шымкента (названия вроде `Shymkent` и `шымкент` уже приведены к одному виду).

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [5]:
# Стандарт
# 4. Скидка 15% на оплаченные заказы дороже 50 000 ₸
big = paid & (amount > 50_000)
loss_all = 0.15 * amount[big].sum()

big_clean = big & (cust != "TEST")          # то же, но без тестового клиента
loss_clean = 0.15 * amount[big_clean].sum()

print(f"4. Со всеми заказами: {fmt(loss_all)} ₸ ({big.sum()} заказов)")
print(f"   Без TEST:          {fmt(loss_clean)} ₸ ({big_clean.sum()} заказов)")
print(f"   Различие: в {loss_all / loss_clean:.1f} раза; заказов TEST под скидку: {(big & (cust == 'TEST')).sum()}")

# 5. Доля заказов, доставленных за 1–2 дня, по городам
fast = (days >= 1) & (days <= 2)     # для NaN сравнение даёт False
known = ~np.isnan(days)              # срок доставки известен
print("\n5. Доля доставок за 1–2 дня (заказы без срока доставки не учитываем):")
for cty in np.unique(city):
    m = (city == cty) & known
    print(f"   {cty:<10} {fast[m].mean():6.1%}   (заказов с известным сроком: {m.sum()})")
print(f"   Заказов без срока доставки: {(~known).sum()}")

# 6. Средний чек: исправляем код AI
# Было (AI):  avg = amount.mean(); print("Средний чек:", round(avg))
avg_bad = amount.mean()

# Стало: только оплаченные заказы и без тестового клиента
ok = paid & (cust != "TEST")
avg = amount[ok].mean()
print(f"\n6. Было: {round(avg_bad)} ₸ | стало: {round(avg)} ₸ | медиана: {round(np.median(amount[ok]))} ₸")
print(f"   Заказов в расчёте: {ok.sum()} из {len(amount)}")
print(f"   Только убрали TEST: {round(amount[cust != 'TEST'].mean())} ₸ | только убрали отменённые: {round(amount[paid].mean())} ₸")

4. Со всеми заказами: 5 702 422 ₸ (127 заказов)
   Без TEST:          1 202 427 ₸ (124 заказов)
   Различие: в 4.7 раза; заказов TEST под скидку: 3

5. Доля доставок за 1–2 дня (заказы без срока доставки не учитываем):
   Актобе      62.8%   (заказов с известным сроком: 393)
   Алматы      66.5%   (заказов с известным сроком: 2123)
   Астана      65.4%   (заказов с известным сроком: 1350)
   Атырау      66.7%   (заказов с известным сроком: 141)
   Караганда   71.0%   (заказов с известным сроком: 514)
   Шымкент     30.7%   (заказов с известным сроком: 782)
   Заказов без срока доставки: 160

6. Было: 20114 ₸ | стало: 14672 ₸ | медиана: 10300 ₸
   Заказов в расчёте: 4901 из 5463
   Только убрали TEST: 14631 ₸ | только убрали отменённые: 20781 ₸


**Ответы [Стандарт]**

4. **Со всеми заказами — 5 702 422 ₸, без TEST — 1 202 427 ₸ (различие в 4,7 раза).** *Менеджеру:* реальные потери от скидки — около 1,2 млн ₸; цифра 5,7 млн раздута тремя тестовыми заказами по 9,99 млн ₸ (их 30 млн ₸ против ≈8 млн ₸ настоящих заказов дороже 50 000 ₸). Выброс в несколько порядков больше типичного заказа полностью определяет сумму.
5. **Доля доставок за 1–2 дня:** Актобе 62,8 %, Алматы 66,5 %, Астана 65,4 %, Атырау 66,7 %, Караганда 71,0 %, **Шымкент 30,7 %**. *Менеджеру:* везде 63–71 % заказов приходят за 1–2 дня, кроме Шымкента, где доля вдвое ниже, — доставку туда нужно разбирать отдельно. (Заказы без срока доставки — 160 штук — в расчёт не берём: «срок неизвестен» не значит «долго».)
6. **Две ошибки:** (1) в среднее попали **три тестовых заказа клиента TEST** по 9 999 990 ₸ — они сильно завышают результат; (2) в среднее попали **отменённые заказы** — это не продажи, средний чек считают по оплаченным (они, наоборот, слегка занижают среднее: если убрать только их, получится 20 781 ₸, поэтому главный вклад даёт TEST). **Правильный средний чек — 14 672 ₸** (было 20 114 ₸). *Менеджеру:* прежний расчёт завышал средний чек на 37 % из-за тестовых заказов; типичный чек ещё ниже — медиана 10 300 ₸. Заметка: в файле также 180 дублей строк (одинаковый `order_id`); без них средний чек — 14 700 ₸.

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [6]:
# Челлендж
import time

# 7. Скорость: цикл for против маски (цикл здесь нужен только для сравнения)
def sum_loop():
    s = 0.0
    for k in range(len(amount)):
        if paid[k]:
            s += amount[k]
    return s

def sum_mask():
    return amount[paid].sum()

assert np.isclose(sum_loop(), sum_mask())    # оба способа дают одну сумму

def avg_time(f, n):
    t0 = time.perf_counter()
    for _ in range(n):
        f()
    return (time.perf_counter() - t0) / n

t_loop = avg_time(sum_loop, 100)
t_mask = avg_time(sum_mask, 10_000)
print(f"7. Цикл: {t_loop * 1e3:.3f} мс | маска: {t_mask * 1e6:.1f} мкс | маска быстрее в {t_loop / t_mask:.0f} раз")

# 8. A/B-тест
def experiment(n=1000, p_a=0.05, p_b=0.06):
    a = (np.random.rand(n) < p_a).sum()     # покупки в группе A
    b = (np.random.rand(n) < p_b).sum()     # покупки в группе B
    return int(a), int(b)

np.random.seed(42)
a, b = experiment()
print(f"\n8. Один эксперимент: A — {a} покупок, B — {b} покупок")

np.random.seed(42)                           # тот же seed: первый из 100 запусков = эксперимент выше
results = np.array([experiment() for _ in range(100)])
b_better = (results[:, 1] > results[:, 0]).sum()
tie = (results[:, 1] == results[:, 0]).sum()
a_better = (results[:, 1] < results[:, 0]).sum()
print(f"   100 повторов: B лучше A в {b_better}, ничья в {tie}, A лучше B в {a_better}")
print(f"   В среднем за запуск: A — {results[:, 0].mean():.1f}, B — {results[:, 1].mean():.1f} покупок")

7. Цикл: 0.677 мс | маска: 8.4 мкс | маска быстрее в 81 раз

8. Один эксперимент: A — 54 покупок, B — 58 покупок
   100 повторов: B лучше A в 84, ничья в 1, A лучше B в 15
   В среднем за запуск: A — 49.6, B — 59.4 покупок


**Ответы [Челлендж]**

7. **Маска быстрее цикла примерно в 70–130 раз** (точное число зависит от компьютера и запуска — смотрите вывод ячейки). *Менеджеру:* на 5 463 заказах разница — доли миллисекунды, но на миллионах заказов цикл `for` превращается в заметное ожидание, а маска остаётся мгновенной, поэтому считаем массивами.
8. **Один эксперимент: A — 54 покупки, B — 58. За 100 повторов B оказалась лучше A в 84 случаях, в одном — ничья, в 15 — лучше оказалась A.** Хотя истинная конверсия у B выше (6 % против 5 %), при 1 000 клиентов в группе тест «ошибается» примерно в каждом шестом запуске: разница в 1 процентный пункт (≈10 покупок) сравнима со случайным разбросом (≈ ±7 покупок на группу). Поэтому один запуск (54 против 58) ничего не доказывает, а для надёжного вывода нужна выборка заметно больше — по стандартной оценке порядка 8 000 клиентов на группу.